# Notebook 02 — Feature Engineering
## JFK Airport Daily Passenger Throughput Forecasting

**Design decisions** are documented in `feature_engineering_decisions.md`.

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, json
from datetime import date, timedelta
import holidays

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

FIGURES = '../reports/figures'
os.makedirs(FIGURES, exist_ok=True)

print('Setup complete.')

Setup complete.


## 2. Load Merged Dataset from Notebook 01

In [ ]:
df = pd.read_csv('../data/processed/jfk_daily_merged.csv', parse_dates=['Date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Columns ({df.shape[1]}): {list(df.columns)}')
df.head(3)

: 

## 3. Drop Unnecessary Columns

Terminal-level columns were used for EDA only — the target `daily_throughput`
is already the sum. We also drop `scheduled_arrivals` and
`total_scheduled_flights` per
[Decision #1](feature_engineering_decisions.md#1-scheduled-departures-only-not-arrivals):
TSA screens only departing passengers, and departures/arrivals are r ≈ 0.99
correlated.

In [ ]:
# Drop terminal columns (EDA only — target is already their sum)
terminal_cols = [c for c in df.columns if 'Terminal' in c]
df = df.drop(columns=terminal_cols)
print(f'Dropped {len(terminal_cols)} terminal columns: {terminal_cols}')

# Drop arrivals and total flights (Decision #1 — departures only)
flight_drop = ['scheduled_arrivals', 'total_scheduled_flights']
flight_drop = [c for c in flight_drop if c in df.columns]
df = df.drop(columns=flight_drop)
print(f'Dropped flight columns: {flight_drop}')

print(f'\nRemaining: {df.shape[1]} columns: {list(df.columns)}')

: 

## 4. Calendar Features (Cyclical Encoding)

Per [Decision #5](feature_engineering_decisions.md#5-cyclical-encoding-for-calendar-features),
we encode `day_of_week`, `month`, and `day_of_year` with sin/cos transforms
to preserve circularity (Sunday↔Monday, December↔January).

We also add `is_weekend` as a binary feature and `quarter` as integer.

In [ ]:
# Extract raw calendar values
df['day_of_week'] = df['Date'].dt.dayofweek      # 0=Mon, 6=Sun
df['month']       = df['Date'].dt.month           # 1–12
df['day_of_year'] = df['Date'].dt.dayofyear       # 1–366
df['quarter']     = df['Date'].dt.quarter          # 1–4
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# Cyclical sin/cos encoding
def cyclical_encode(df, col, period):
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / period)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / period)
    return df

df = cyclical_encode(df, 'day_of_week', 7)
df = cyclical_encode(df, 'month', 12)

days_in_year = df['Date'].dt.is_leap_year.map({True: 366, False: 365})
df = cyclical_encode(df, 'day_of_year', days_in_year)

# Drop raw integer versions (keep only sin/cos) 
# Drop quarter 
# Keep only is_weekend
df = df.drop(columns=['day_of_week', 'month', 'day_of_year', 'quarter'])

calendar_features = ['quarter', 'is_weekend',
                     'day_of_week_sin', 'day_of_week_cos',
                     'month_sin', 'month_cos',
                     'day_of_year_sin', 'day_of_year_cos']
print(f'Calendar features ({len(calendar_features)}): {calendar_features}')

: 